#  Quantum Threat Visualizations

## Understanding Cryptographic Risk and Migration

---

### [!] DISCLAIMER ON UNCERTAINTY

> **There is no reliable date for a cryptographically relevant quantum computer.**
>
> - There is **no reliable date** for when cryptographically relevant quantum computers (CRQCs) will exist
> - Resource estimates and hardware roadmaps depend on assumptions and are not reliable attack-date forecasts
> - These visualizations distinguish cryptographic effects and migration policy from hardware predictions
> - The purpose is to help organizations make **informed decisions** about migration timing

---

###  Learning Objectives

After completing this notebook, you will understand:

1. The **"Harvest Now, Decrypt Later" (HNDL)** threat model
2. **Mosca's Theorem** for cryptographic risk assessment
3. The difference between **symmetric** (AES) and **asymmetric** (RSA) quantum vulnerability
4. How to interpret **quantum resource estimates** in context
5. Why migration should begin **now** despite uncertain timelines

---

## Part 1: The Harvest Now, Decrypt Later (HNDL) Threat

### 1.1 Understanding the Threat Model

The HNDL attack is deceptively simple:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    HARVEST NOW, DECRYPT LATER                            │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│   TODAY                         IF CAPABILITIES CHANGE                    │
│   ───────────                          ────────────────                  │
│                                                                          │
│   ┌─────────────┐      Store       ┌─────────────────┐                  │
│   │ Encrypted   │ ───────────────► │ Encrypted Data  │                  │
│   │ Traffic     │                  │ Archive         │                  │
│   └─────────────┘                  └────────┬────────┘                  │
│         │                                   │                            │
│         │ Intercepted by                    │ Quantum                    │
│         │ adversary                         │ Computer                   │
│         │                                   ▼                            │
│         │                          ┌─────────────────┐                  │
│         │                          │ Decrypted Data! │                  │
│         │                          │ • State secrets │                  │
│         │                          │ • Medical records│                  │
│         │                          │ • Financial data │                  │
│         │                          └─────────────────┘                  │
│                                                                          │
│   RSA-2048 encryption               Shor's Algorithm                    │
│   not broken by current machines                   could break it with capable hardware                  │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

### 1.2 Why This Matters Now

| Data type | Why confidentiality lifetime matters |
|-----------|------------------------------------|
| State secrets, health records, trade secrets, and private communications | Exposure could remain harmful for many years |

**Key Insight:** HNDL is a future-decryption risk model, not a claim that current quantum computers can decrypt captured data. The arrival date of capable hardware is unknown.

In [1]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from datetime import date
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
%matplotlib inline

# Use a clean style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

# Color palette
COLORS = {
    'critical': '#e74c3c',
    'warning': '#f39c12',
    'safe': '#27ae60',
    'info': '#3498db',
    'dark': '#2c3e50',
    'light': '#ecf0f1'
}

print("[OK] Visualization environment ready!")

[OK] Visualization environment ready!


---

## Part 2: Mosca's Theorem — The Risk Inequality

### 2.1 Mathematical Foundation

[Michele Mosca](https://globalriskinstitute.org/publication/quantum-threat-timeline-report-2022/) formalized the quantum cryptographic risk with a simple inequality:

$$\boxed{X + Y > Z \implies \text{Immediate Action Required}}$$

Where:

| Variable | Definition | Typical Range |
|----------|------------|---------------|
| $X$ | **Shelf Life** — How long must the data remain secret? | 5-50+ years |
| $Y$ | **Migration Time** — How long to migrate systems to PQC? | 2-15 years |
| $Z$ | **Collapse Time** — When will CRQCs exist? | Unknown (5-30+ years) |

### 2.2 The Logic

- If $(X + Y) > Z$: Your secrets will be exposed before you finish migrating. **ACT NOW.**
- If $(X + Y) \leq Z$: You have time, but should still monitor and plan.

### 2.3 Why Uncertainty Favors Early Action

Since $Z$ is **unknown**, organizations must consider:
- Worst-case scenario: $Z$ could be 5-10 years (optimistic quantum progress)
- Large systems ($Y = 10-15$ years) have no margin for error

In [ ]:
def visualize_mosca_theorem(shelf_life, migration_time, collapse_scenarios):
    """
    Visualize Mosca's Theorem: X + Y > Z inequality.
    
    Args:
        shelf_life: X - How long data must remain secret (years)
        migration_time: Y - Time to migrate to PQC (years)
        collapse_scenarios: dict of {name: Z value} for different scenarios
    """
    fig, ax = plt.subplots(figsize=(14, 8))
    
    current_year = date.today().year
    combined_time = shelf_life + migration_time
    
    # Y positions for bars
    y_positions = {
        'shelf_life': 4,
        'migration': 3,
        'combined': 2
    }
    
    bar_height = 0.6
    
    # Draw Shelf Life bar (X)
    ax.barh(y_positions['shelf_life'], shelf_life, height=bar_height, 
            left=current_year, color=COLORS['info'], alpha=0.8,
            label=f'X: Data Shelf Life ({shelf_life} years)')
    ax.text(current_year + shelf_life/2, y_positions['shelf_life'], 
            f'X = {shelf_life} years', ha='center', va='center', 
            fontsize=12, fontweight='bold', color='white')
    
    # Draw Migration Time bar (Y)
    ax.barh(y_positions['migration'], migration_time, height=bar_height,
            left=current_year, color=COLORS['warning'], alpha=0.8,
            label=f'Y: Migration Time ({migration_time} years)')
    ax.text(current_year + migration_time/2, y_positions['migration'],
            f'Y = {migration_time} years', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
    
    # Draw Combined bar (X + Y)
    ax.barh(y_positions['combined'], combined_time, height=bar_height,
            left=current_year, color=COLORS['dark'], alpha=0.8,
            label=f'X + Y: Combined ({combined_time} years)')
    ax.text(current_year + combined_time/2, y_positions['combined'],
            f'X + Y = {combined_time} years', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
    
    # Draw collapse scenario lines (Z)
    scenario_y = 0.8
    scenario_colors = {'Optimistic': COLORS['critical'], 
                       'Moderate': COLORS['warning'], 
                       'Conservative': COLORS['safe']}
    
    for i, (name, z_value) in enumerate(collapse_scenarios.items()):
        year = current_year + z_value
        color = scenario_colors.get(name, COLORS['info'])
        
        # Vertical line for Z
        ax.axvline(x=year, color=color, linestyle='--', linewidth=2.5, alpha=0.8)
        
        # Label
        ax.text(year, 5.2 - i*0.4, f'Z = {z_value}yr ({name})', 
                ha='center', va='bottom', fontsize=10, color=color, fontweight='bold')
        
        # Risk status
        if combined_time > z_value:
            status = "[!] RISK"
            status_color = COLORS['critical']
        else:
            status = "[OK] OK"
            status_color = COLORS['safe']
        
        ax.text(year, scenario_y - i*0.3, status, 
                ha='center', va='top', fontsize=9, color=status_color, fontweight='bold')
    
    # Formatting
    ax.set_xlim(current_year - 2, current_year + max(combined_time, max(collapse_scenarios.values())) + 5)
    ax.set_ylim(-0.5, 6)
    ax.set_xlabel('Year', fontsize=14)
    ax.set_yticks([y_positions['shelf_life'], y_positions['migration'], y_positions['combined']])
    ax.set_yticklabels(['Data Shelf Life (X)', 'Migration Time (Y)', 'Combined (X+Y)'], fontsize=12)
    ax.set_title("Mosca's Theorem: When Should You Worry?\n" + 
                 r"If $X + Y > Z$, then migration must begin NOW", 
                 fontsize=16, fontweight='bold', pad=20)
    
    # Add legend
    ax.legend(loc='lower right', fontsize=10)
    
    # Add grid
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add annotation box
    textstr = ('Mosca\'s Inequality:\n'
               f'X (Shelf Life) = {shelf_life} years\n'
               f'Y (Migration) = {migration_time} years\n'
               f'X + Y = {combined_time} years\n\n'
               'If X + Y > Z: ACT NOW!')
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n" + "="*60)
    print("MOSCA'S THEOREM RISK ANALYSIS")
    print("="*60)
    print(f"\nYour combined timeline: X + Y = {combined_time} years")
    print(f"\nScenario Analysis:")
    for name, z in collapse_scenarios.items():
        if combined_time > z:
            print(f"  {name} (Z={z}): [!]  CRITICAL - Start migration immediately!")
        else:
            margin = z - combined_time
            print(f"  {name} (Z={z}): [OK] Safe with {margin} year margin")


# =============================================================================
# MOSCA'S THEOREM VISUALIZATION
# =============================================================================

print("="*60)
print("MOSCA'S THEOREM: QUANTUM RISK ASSESSMENT")
print("="*60)

#  INTERACTIVE: Adjust these parameters for your organization!
SHELF_LIFE = 25        # X: How long must your secrets remain secret? (years)
MIGRATION_TIME = 10    # Y: How long will migration take? (years)

# Collapse time scenarios (Z)
COLLAPSE_SCENARIOS = {
    'Optimistic': 10,     # Aggressive quantum progress
    'Moderate': 15,       # Steady progress
    'Conservative': 25    # Significant technical hurdles
}

visualize_mosca_theorem(SHELF_LIFE, MIGRATION_TIME, COLLAPSE_SCENARIOS)

---

## Part 3: Symmetric vs Asymmetric Vulnerability

### 3.1 The Critical Distinction

Not all cryptography is equally vulnerable to quantum attacks:

| Algorithm Type | Quantum algorithm | Effect and caveat |
|---------------|-------------------|-------------------|
| **Public key** (RSA, ECC, DH) | Shor's algorithm | Vulnerable in principle on a sufficiently capable fault-tolerant machine; current machines cannot do this at cryptographic scale |
| **Symmetric** (AES) | Grover's algorithm | Quadratic speedup in an idealized search model; the result is not a direct real-world attack time |
| **Hash functions** | Grover's algorithm | Affects preimage search; collision security has a different analysis |

### 3.2 Mathematical Explanation

**Grover's Algorithm (Symmetric):**
- Gives a quadratic speedup for unstructured search in an idealized query model.
- AES-256 has about $2^{128}$ idealized quantum queries of key-search complexity.
- Query complexity is not a direct runtime or a complete assessment of a real system.

**Shor's Algorithm (Asymmetric):**
- Solves discrete log and factoring in polynomial time
- RSA-2048 is vulnerable in principle, but no current machine can factor it
- Larger RSA keys do not remove this asymptotic vulnerability; resource needs still depend on the key size and implementation

In [ ]:
def visualize_security_degradation():
    """Show qualitative algorithm effects without assigning a threat date."""
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.axis('off')
    ax.set_title('What a Large Fault-Tolerant Quantum Computer Could Change',
                 fontsize=17, fontweight='bold', pad=24)

    rows = [
        ('Public-key factoring and discrete logarithms',
         'Shor's algorithm threatens RSA, Diffie–Hellman, and elliptic-curve cryptography in principle.',
         COLORS['critical']),
        ('Symmetric-key search',
         'Grover gives a quadratic query speedup in an idealized model; practical costs depend on the attack and implementation.',
         COLORS['warning']),
        ('Post-quantum standards',
         'ML-KEM, ML-DSA, and SLH-DSA are designed to resist known classical and quantum attacks.',
         COLORS['safe']),
    ]
    for i, (heading, detail, color) in enumerate(rows):
        y = 0.77 - i * 0.27
        ax.add_patch(FancyBboxPatch((0.04, y - 0.16), 0.92, 0.22,
                                    boxstyle='round,pad=0.012', facecolor=color,
                                    alpha=0.15, edgecolor=color, linewidth=1.5,
                                    transform=ax.transAxes))
        ax.text(0.07, y, heading, transform=ax.transAxes, color=color,
                fontsize=13, fontweight='bold', va='center')
        ax.text(0.07, y - 0.07, detail, transform=ax.transAxes,
                color=COLORS['dark'], fontsize=11, va='center', wrap=True)

    ax.text(0.05, 0.04,
            'This is a conceptual comparison, not a forecast of when such hardware will exist.',
            transform=ax.transAxes, fontsize=10, color=COLORS['dark'])
    plt.tight_layout()
    plt.show()


---

## Part 4: Quantum Resource Estimates — Context Matters

A physical-qubit count alone does not measure whether a machine can run Shor's algorithm at cryptographic scale. A useful estimate must specify the circuit, error-correction scheme, hardware assumptions, and target runtime.

| Reference | RSA-2048 estimate | Scope |
|-----------|-------------------|-------|
| Gidney & Ekerå (2021) | About 20 million physical qubits for an 8-hour computation | One architecture and error-correction model; not a universal threshold |

Logical qubits are error-corrected units; physical qubits are hardware devices used to encode them. The conversion depends on error rates and the chosen code. Current machines cannot factor cryptographic-size RSA moduli.

This estimate does not predict when a machine with those resources will exist. See the paper cited below for its assumptions.


In [ ]:
def visualize_qubit_progress():
    """Show one published resource estimate, without projecting qubit growth."""
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.barh(['Gidney & Ekerå (2021)'], [20], color=COLORS['info'], alpha=0.85)
    ax.text(10, 0, '20 million physical qubits',
            va='center', ha='center', color='white', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 24)
    ax.set_xlabel('Estimated physical qubits (millions)')
    ax.set_title('One Model-Dependent RSA-2048 Resource Estimate', fontweight='bold')
    ax.grid(axis='x', alpha=0.25)
    ax.text(0.02, 0.03,
            'The paper modeled an 8-hour runtime. This is not a current capability, universal threshold, or forecast.\n'
            'Logical and physical qubit counts are different quantities.',
            transform=ax.transAxes, fontsize=9, color=COLORS['dark'])
    plt.tight_layout()
    plt.show()


---

## Part 5: Migration Planning

Migration dates are policy milestones, not predictions of quantum hardware. NIST's transition guidance is currently an initial public draft and targets removing quantum-vulnerable public-key algorithms from its standards by 2035, with high-risk systems transitioning earlier.

A practical planning sequence is:

1. Inventory where RSA, Diffie–Hellman, and elliptic-curve algorithms are used.
2. Prioritize systems that protect data requiring long-term confidentiality, including HNDL exposure.
3. Check current NIST standards and relevant sector or jurisdiction requirements.
4. Plan interoperability tests and crypto-agility before deployment.
5. Use maintained cryptographic providers appropriate to the deployment; do not use this notebook's toy code in production.

See [NIST IR 8547](https://csrc.nist.gov/pubs/ir/8547/ipd) for the draft transition plan and [NIST's PQC overview](https://csrc.nist.gov/projects/post-quantum-cryptography) for current status.


In [ ]:
def create_summary_dashboard():
    """Summarize published standards, work in progress, and policy status."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor('white')
    fig.suptitle('Post-Quantum Cryptography Status (October 2026)',
                 fontsize=17, fontweight='bold')

    cards = [
        ('FINAL NIST STANDARDS', 'FIPS 203: ML-KEM
FIPS 204: ML-DSA
FIPS 205: SLH-DSA', COLORS['safe']),
        ('SELECTED; IN DEVELOPMENT', 'FN-DSA / Falcon: intended FIPS 206
HQC: intended FIPS 207', COLORS['info']),
        ('QUANTUM HARDWARE', 'Current machines cannot factor
cryptographic-size RSA keys.
No reliable attack date is known.', COLORS['warning']),
        ('MIGRATION POLICY', 'NIST targets transition by 2035.
This is a policy milestone,
not a Q-Day forecast.', COLORS['critical']),
    ]
    for ax, (heading, detail, color) in zip(axes.flat, cards):
        ax.set_facecolor(color)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(color)
        ax.text(0.06, 0.76, heading, transform=ax.transAxes,
                fontsize=12, fontweight='bold', color='white', va='center')
        ax.text(0.06, 0.43, detail, transform=ax.transAxes,
                fontsize=12, color='white', va='center', linespacing=1.5)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


---

##  References

1. Mosca, M. (2018). "Cybersecurity in an Era with Quantum Computers: Will We Be Ready?" *IEEE Security & Privacy*, 16(5), 38-41.

2. Gidney, C., & Ekerå, M. (2021). "How to factor 2048 bit RSA integers in 8 hours using 20 million noisy qubits." *Quantum*, 5, 433.

3. NIST, [Post-Quantum Cryptography](https://csrc.nist.gov/projects/post-quantum-cryptography), [IR 8547 transition draft](https://csrc.nist.gov/pubs/ir/8547/ipd), and [SP 800-227 final KEM guidance](https://csrc.nist.gov/pubs/sp/800/227/final).

4. Global Risk Institute. "Quantum Threat Timeline Report." [Annual Reports](https://globalriskinstitute.org/)

5. IBM Quantum Roadmap. [IBM Quantum Network](https://www.ibm.com/quantum)

---

## --> Next Steps

Continue to **Notebook 04: Post-Quantum Standards (NIST)** to learn:
- ML-KEM (Kyber) key encapsulation
- ML-DSA (Dilithium) digital signatures
- The Learning With Errors (LWE) problem
- Practical implementation considerations

---

*Created for educational purposes. All code is provided under MIT License.*